# EPA and Coal Plants: Group Assignment 1
**Data & Decisions** · Team members: _<add names>_

Input: `data/EPA_CoalUnitCharacteristics2022.xls` (EPA 2022 file from bCourses, unchanged).
All random draws use `SEED = 2022`. Re-running the notebook top to bottom reproduces every table, and the selected unit IDs are also saved to `outputs/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 100)

SEED = 2022
DATA = Path("data/EPA_CoalUnitCharacteristics2022.xls")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

POLLUTANTS = ["2022 SO2 (tons)", "2022 NOX (tons)", "2022 CO2 (tons)"]
STATE = "State Name"

## Part 1: Define the frame and calculate benchmarks

### 1a. Rows, missing-data rule, plants, states, and units per state

Part 1a asks for five things, which we report in order below:

| | Required | Where |
|---|---|---|
| (i) | Rows before and after the missing-data rule | Step 1-3 |
| (ii) | Distinct plants, counted by `ORISPL` | Step 4 |
| (iii) | States represented in the analysis frame | Step 4 |
| (iv) | **Unit key is `ORISPL` + `Boiler ID`, and the combined identifiers are unique** | Step 5 |
| (v) | Number of units in each state | Step 6 |

Item (iv) is the identifier we carry through Parts 2-4 to track which units were sampled, so we establish
it here, in Part 1a, before any sample is drawn.

**Step 1. Load the file and build the unit key.**

In [2]:
# Rows 0-2 of the sheet are a title block; row 3 holds the column headers.
# Boiler ID is read as text so IDs like "001" keep their leading zeros and
# non-numeric IDs like "SN-01" are not coerced to NaN.
raw = pd.read_excel(DATA, header=3, dtype={"Boiler ID": str})

key = ["ORISPL", "Boiler ID"]                                    # the unit key specified in 1a
raw["unit_id"] = raw["ORISPL"].astype(str) + "-" + raw["Boiler ID"]   # readable label for the same key

print("Rows in file (one row per generating unit):", len(raw))
print("\nMissing values per pollutant:")
print(raw[POLLUTANTS].isna().sum())
print("\nReported zeros per pollutant (these are kept, not treated as missing):")
print((raw[POLLUTANTS] == 0).sum())

Rows in file (one row per generating unit): 452

Missing values per pollutant:
2022 SO2 (tons)    0
2022 NOX (tons)    0
2022 CO2 (tons)    4
dtype: int64

Reported zeros per pollutant (these are kept, not treated as missing):
2022 SO2 (tons)    0
2022 NOX (tons)    0
2022 CO2 (tons)    0
dtype: int64


**Step 2. Apply the missing-data rule and record the excluded units.**

In [3]:
missing_any = raw[POLLUTANTS].isna().any(axis=1)

excluded = raw.loc[missing_any, ["unit_id"] + key + ["Plant Name", STATE] + POLLUTANTS].copy()
excluded["reason"] = raw.loc[missing_any, POLLUTANTS].isna().apply(
    lambda r: "missing " + ", ".join(c for c in POLLUTANTS if r[c]), axis=1)
excluded.to_csv(OUT / "excluded_units.csv", index=False)
excluded

,unit_id,ORISPL,Boiler ID,Plant Name,State Name,2022 SO2 (tons),2022 NOX (tons),2022 CO2 (tons),reason
186,50835-1,50835,1,TES Filer City Station,Michigan,129,656,NaN,missing 2022 CO2 (tons)
187,50835-2,50835,2,TES Filer City Station,Michigan,130,550,NaN,missing 2022 CO2 (tons)
322,50776-1,50776,1,Panther Creek Energy Facility,Pennsylvania,172,188,NaN,missing 2022 CO2 (tons)
323,50776-2,50776,2,Panther Creek Energy Facility,Pennsylvania,219,171,NaN,missing 2022 CO2 (tons)


**Step 3. Build the analysis frame:** units reporting all three pollutants. Missing values are dropped, never replaced with zero.

In [4]:
frame = raw.loc[~missing_any].reset_index(drop=True)

assert frame[POLLUTANTS].notna().all().all()
print("Rows before missing-data rule:", len(raw))
print("Rows excluded:", int(missing_any.sum()))
print("Rows in analysis frame:", len(frame))

Rows before missing-data rule: 452
Rows excluded: 4
Rows in analysis frame: 448


**Step 4. (i)-(iii) Row counts, distinct plants, and states represented.**

In [5]:
summary_1a = pd.Series({
    "Rows before missing-data rule": len(raw),
    "Rows excluded (missing a pollutant)": int(missing_any.sum()),
    "Rows in analysis frame": len(frame),
    "Distinct plants (counted by ORISPL)": frame["ORISPL"].nunique(),
    "States represented": frame[STATE].nunique(),
}, name="value")
summary_1a.to_frame()

,value
Rows before missing-data rule,452
Rows excluded (missing a pollutant),4
Rows in analysis frame,448
Distinct plants (counted by ORISPL),218
States represented,39


**Step 5. (iv) The unit key, and the check that combined identifiers are unique.**

1a specifies `ORISPL` for plants and `ORISPL` + `Boiler ID` for units. We verify uniqueness on the
**column pair itself**, not only on the concatenated `unit_id` string, because the pair is the stricter
test: a concatenation can mask a collision by joining two different pairs into the same string
(e.g. `ORISPL=12, Boiler="3-4"` and `ORISPL=123, Boiler="4"` both give `"12-3-4"`).

That is not a hypothetical concern in this file. Boiler IDs are **not** all numeric: they include values
such as `SN-01`, `U1B`, `AA001`, `BB04` and `3BLR`, and four of them already contain hyphens, the exact
character we join on. So we (a) read `Boiler ID` as text, (b) treat the pair as the key, and (c) use
`unit_id` only as a convenient label, after confirming the two agree. We also confirm `ORISPL` alone is
*not* unique, which is the expected signature of multi-unit plants and the reason the Boiler ID is needed.

In [6]:
dup_pairs = frame.duplicated(subset=key).sum()
id_check = pd.Series({
    "Rows in analysis frame": len(frame),
    "Distinct (ORISPL, Boiler ID) pairs": len(frame.drop_duplicates(subset=key)),
    "Duplicate pairs found": dup_pairs,
    "(ORISPL, Boiler ID) uniquely identifies a unit": dup_pairs == 0,
    "Distinct ORISPL (plants) alone": frame["ORISPL"].nunique(),
    "ORISPL alone is unique? (expect False: plants have several units)": frame["ORISPL"].is_unique,
    "Concatenated unit_id also unique": frame["unit_id"].is_unique,
    "Pair count == unit_id count (no collisions from concatenation)":
        len(frame.drop_duplicates(subset=key)) == frame["unit_id"].nunique(),
}, name="value")

assert dup_pairs == 0
assert len(frame.drop_duplicates(subset=key)) == frame["unit_id"].nunique()
id_check.to_frame()

,value
Rows in analysis frame,448
"Distinct (ORISPL, Boiler ID) pairs",448
Duplicate pairs found,0
"(ORISPL, Boiler ID) uniquely identifies a unit",True
Distinct ORISPL (plants) alone,218
ORISPL alone is unique? (expect False: plants have several units),False
Concatenated unit_id also unique,True
Pair count == unit_id count (no collisions from concatenation),True


In [7]:
# Supporting evidence for the two claims above.
# (a) Plants contributing the most units: ORISPL alone cannot identify a unit.
display(frame.groupby(["ORISPL", "Plant Name"]).size().rename("units at plant")
             .sort_values(ascending=False).head(8).to_frame())

# (b) Boiler IDs are free-form text, and some contain the hyphen we join on.
print("Non-numeric Boiler IDs:", (~frame["Boiler ID"].str.fullmatch(r"\d+")).sum(), "of", len(frame))
print("Boiler IDs containing a hyphen:", frame["Boiler ID"].str.contains("-").sum())
print("Examples:", frame.loc[~frame["Boiler ID"].str.fullmatch(r"\d+"), "Boiler ID"].unique()[:12].tolist())

,,units at plant
ORISPL,Plant Name,
3407,Kingston,9
1379,Shawnee,9
887,Joppa Steam,6
983,Clifty Creek,6
2712,Roxboro,6
2876,Kyger Creek,5
6113,Gibson,5
1743,St. Clair,4


Non-numeric Boiler IDs: 93 of 448
Boiler IDs containing a hyphen: 4
Examples: ['U1B', 'U2B', 'TS3', 'SN-01', 'C1', 'C2', 'C3', 'H1', 'H2', 'BB04', 'B2', '1A']


**Step 6. (v) Number of units in each state.** These counts are the $N_h$ used for the weights in Part 4b.

In [8]:
state_counts = (frame[STATE].value_counts().rename("N_h (units in frame)")
                .rename_axis(STATE).sort_index().to_frame())
state_counts.to_csv(OUT / "units_per_state.csv")
print("Total units across all states:", state_counts.iloc[:, 0].sum(), "(should equal the frame size, 448)")
state_counts

Total units across all states: 448 (should equal the frame size, 448)


,N_h (units in frame)
State Name,
Alabama,7
Arizona,9
Arkansas,7
Colorado,11
Delaware,1
Florida,10
Georgia,9
Illinois,24
Indiana,32


### 1b. Full-frame benchmarks (mean tons per unit, 2022)

In [9]:
az = frame[frame[STATE] == "Arizona"]

benchmarks = pd.DataFrame({
    "United States (N=%d)" % len(frame): frame[POLLUTANTS].mean(),
    "Arizona (N=%d)" % len(az): az[POLLUTANTS].mean(),
}).rename(index=lambda c: c.replace("2022 ", "Mean 2022 "))
benchmarks.index.name = "Full-frame mean, tons per unit"
benchmarks

,United States (N=448),Arizona (N=9)
"Full-frame mean, tons per unit",,
Mean 2022 SO2 (tons),"1,860.48",808.67
Mean 2022 NOX (tons),"1,293.89","1,275.78"
Mean 2022 CO2 (tons),"2,111,480.54","1,825,836.89"


### 1c. Which population do these benchmarks describe?

The benchmarks describe exactly one population: **the 448 coal-fired generating units in this analysis frame**, meaning the units listed in the EPA's 2022 NEEDS v621 coal unit file that report all three of SO2, NOX and CO2 for 2022. Within that population the means are not estimates at all. They are the true values, computed from a census of the frame, so they carry no sampling error and need no confidence interval. That is why we can use them later as the yardstick against which the sampled estimates are judged.

**Having every row in the file does not establish coverage of every US coal-fired unit.** A census of a frame is only as complete as the frame. Two separate gaps sit between these 448 units and "all US coal units":

1. *Frame coverage.* The file is EPA's list, built for EPA's programs and vintage-stamped NEEDS v621 (October 2022). Units that fall outside whatever inclusion rule EPA applied, for instance very small units, units below a reporting threshold, units that retired or came online mid-year, or units classified under a different primary fuel, would never appear as a row no matter how complete the file is internally. Nothing inside the file can reveal a unit the file omits.
2. *Our exclusions.* We dropped 4 units for missing 2022 CO2: both units at TES Filer City Station (ORISPL 50835, Boiler IDs 1 and 2) in Michigan and both units at Panther Creek Energy Facility (ORISPL 50776, Boiler IDs 1 and 2) in Pennsylvania. These are real, operating units that reported SO2 and NOX. Excluding them is a choice we made for a consistent analysis frame, not evidence that the units do not exist.

Note that the missing values are plainly not zeros. All four units reported substantial SO2 and NOX, so they clearly burned fuel and clearly emitted CO2. The value is unreported, not zero. Had we replaced the blanks with 0, we would have invented four zero-CO2 coal units and pulled the national CO2 mean down by about 18,700 tons per unit (2,111,481 to 2,092,795), which is the mechanical consequence of the error rather than any fact about emissions. Our rule also means two whole plants drop out, so state unit counts fall by 2 in Michigan (19 to 17) and 2 in Pennsylvania (27 to 25), and those reduced counts are what every later weight is built from.

So the file and our exclusions **do** establish: what these 448 units emitted in 2022, and therefore an exact per-unit average for this frame. They **do not** establish: that the frame is all US coal units, that results generalize to units EPA did not list, or anything about the 4 excluded units' CO2.

## Part 2: National simple random sample (n = 150)
### 2a. Draw the sample

In [10]:
# SRS without replacement: pandas DataFrame.sample with replace=False (the default), seeded.
srs = frame.sample(n=150, replace=False, random_state=SEED)
srs[["unit_id", "ORISPL", "Boiler ID", "Plant Name", STATE] + POLLUTANTS].to_csv(OUT / "srs_sample.csv", index=False)

assert len(srs) == 150
assert srs["unit_id"].nunique() == 150
assert srs["unit_id"].isin(frame["unit_id"]).all()
print("Method: simple random sample without replacement, pandas DataFrame.sample(n=150, random_state=%d)" % SEED)

srs_check = pd.Series({
    "Units selected": len(srs),
    "Distinct (ORISPL, Boiler ID) pairs": len(srs.drop_duplicates(subset=key)),
    "Exactly 150 distinct units": len(srs.drop_duplicates(subset=key)) == 150,
    "All selected units eligible (present in frame)": srs["unit_id"].isin(frame["unit_id"]).all(),
    "No unit selected twice (sampling without replacement)": srs["unit_id"].is_unique,
}, name="value")
srs_check.to_frame()

Method: simple random sample without replacement, pandas DataFrame.sample(n=150, random_state=2022)


,value
Units selected,150
"Distinct (ORISPL, Boiler ID) pairs",150
Exactly 150 distinct units,True
All selected units eligible (present in frame),True
No unit selected twice (sampling without replacement),True


**Selected unit identifiers, Part 2a.** All 150 selected units, keyed by `ORISPL` + `Boiler ID`. Also saved to `outputs/srs_sample.csv`.

In [11]:
srs_ids = (srs[["ORISPL", "Boiler ID", "unit_id", "Plant Name", STATE]]
           .sort_values(key).reset_index(drop=True))
srs_ids.index += 1
with pd.option_context("display.max_rows", 160):
    display(srs_ids)

,ORISPL,Boiler ID,unit_id,Plant Name,State Name
1,3,5,3-5,Barry,Alabama
2,113,3,113-3,Cholla,Arizona
3,130,1,130-1,Cross,South Carolina
4,136,1,136-1,Seminole (136),Florida
5,160,3,160-3,Apache Station,Arizona
6,298,LIM1,298-LIM1,Limestone,Texas
7,470,3,470-3,Comanche (470),Colorado
8,564,2,564-2,Curtis H. Stanton Energy Center,Florida
9,602,1,602-1,Brandon Shores,Maryland
10,663,B2,663-B2,Deerhaven,Florida


### 2b. SRS means vs national benchmarks

_Answer after the table below._

In [12]:
us_bench = frame[POLLUTANTS].mean()
srs_mean = srs[POLLUTANTS].mean()

part2b = pd.DataFrame({
    "US benchmark (N=448)": us_bench,
    "SRS mean (n=150)": srs_mean,
    "Difference (SRS - benchmark)": srs_mean - us_bench,
    "% difference": 100 * (srs_mean - us_bench) / us_bench,
})
part2b.index.name = "tons per unit"
part2b

,US benchmark (N=448),SRS mean (n=150),Difference (SRS - benchmark),% difference
tons per unit,,,,
2022 SO2 (tons),"1,860.48","1,548.34",-312.14,-16.78
2022 NOX (tons),"1,293.89","1,231.49",-62.40,-4.82
2022 CO2 (tons),"2,111,480.54","2,006,362.91","-105,117.63",-4.98


**Discussion.** The sample means land below the benchmarks on all three pollutants: SO2 is 1,548.34 vs 1,860.48 tons (-16.8%), NOX is 1,231.49 vs 1,293.89 tons (-4.8%), and CO2 is 2,006,363 vs 2,111,481 tons (-5.0%).

**No, a valid random sample does not have to reproduce the benchmarks exactly, and it almost never will.** What simple random sampling guarantees is a property of the procedure, not of any one draw: the sample mean is an *unbiased* estimator, so averaged over all possible samples of 150 it equals the frame mean. Any single sample is one realization scattered around that target, and the scatter is sampling error, not a mistake. We drew 150 of 448 units, so each draw omits roughly two-thirds of the frame; a different seed would give different numbers.

The size of the gaps is also informative about the data rather than about our sampling. SO2 misses by 16.8% while NOX and CO2 miss by about 5%. That ordering reflects how skewed each variable is. CO2 tracks fuel burned and is fairly even across units, so a sample of 150 pins it down well. SO2 depends on coal sulfur content and on whether a scrubber is installed, so a handful of uncontrolled units emit enormous amounts while scrubbed units emit very little. Whether a few of those extreme units happen to be drawn swings the SO2 mean far more than the CO2 mean. The estimator is unbiased for all three, but it is noisier for the heavy-tailed pollutant.

### 2c. Arizona units in the national SRS

In [13]:
srs_az = srs[srs[STATE] == "Arizona"]
az_bench = az[POLLUTANTS].mean()
print("Arizona units selected in SRS:", len(srs_az), "of", len(az))
print(srs_az[["unit_id", "Plant Name"]].to_string(index=False))

if len(srs_az) > 0:
    part2c = pd.DataFrame({
        "AZ benchmark (N=%d)" % len(az): az_bench,
        "AZ mean in SRS (n=%d)" % len(srs_az): srs_az[POLLUTANTS].mean(),
    })
    part2c["Difference"] = part2c.iloc[:, 1] - part2c.iloc[:, 0]
    display(part2c)
else:
    print("No Arizona units were selected, so no Arizona mean can be calculated from the SRS.")

Arizona units selected in SRS: 3 of 9
unit_id                       Plant Name
  160-3                   Apache Station
  113-3                           Cholla
 8223-2 Springerville Generating Station


,AZ benchmark (N=9),AZ mean in SRS (n=3),Difference
2022 SO2 (tons),808.67,969.00,160.33
2022 NOX (tons),"1,275.78","1,539.33",263.56
2022 CO2 (tons),"1,825,836.89","1,707,136.67","-118,700.22"


In [14]:
# Context for 2c: Arizona's expected count under SRS and the chance of drawing zero AZ units.
from math import comb
N, n, N_az = len(frame), 150, len(az)
print("Expected AZ units in SRS: %.2f" % (n * N_az / N))
print("P(zero AZ units): %.4f" % (comb(N - N_az, n) / comb(N, n)))

Expected AZ units in SRS: 3.01
P(zero AZ units): 0.0245


**Discussion.** Three of Arizona's 9 units were drawn: Apache Station unit 3, Cholla unit 3, and Springerville unit 2. Their means are 969.00 tons SO2, 1,539.33 tons NOX and 1,707,137 tons CO2, against Arizona benchmarks of 808.67, 1,275.78 and 1,825,837 tons. SO2 is 20% high, NOX is 21% high, CO2 is 6.5% low.

**Why national random sampling gives limited subgroup information.** A national SRS is designed to represent the nation, and it treats Arizona's 9 units like any other 9 of the 448. Three consequences follow, and all three are visible here:

1. *The subgroup sample size is not chosen, it is a random outcome.* The expected number of Arizona units is 150 x 9/448 = 3.01, but the realized count could have been 1, or 6. We happened to get 3.
2. *Coverage is not guaranteed at all.* The probability of drawing zero Arizona units is C(439,150)/C(448,150) = 2.45%. One state in forty of this size would come up empty in a given draw, and then no Arizona mean could be computed from the sample. The design makes no promise here. For the genuinely small states the risk is severe: Delaware, South Dakota and Washington have a single unit each, so each appears with probability 150/448 = 33% and is missed two times in three.
3. *Even when the subgroup appears, 3 units is a very thin basis for an estimate.* The same unbiasedness argument from 2b applies to the Arizona mean, but with n=3 the variance is large, which is exactly why the AZ gaps (20%) are wider than the national ones (5%).

The deeper point is that a design optimized for one estimand is not automatically good for another. Precision at the national level comes from the full 150 units; precision for Arizona comes only from however many Arizona units happened to be drawn. If subgroup estimates are a goal, the subgroup sample size has to be designed rather than left to chance, which is what Part 3 does.

## Part 3: Sample within states
### 3a. Five units per state (or all units if fewer than five)

In [15]:
# One seeded generator drives every state's draw, so each state's draw is independent
# and the whole set is reproducible. Each state is sampled without replacement.
rng = np.random.default_rng(SEED)
strat = pd.concat(
    [g.sample(n=min(5, len(g)), replace=False, random_state=rng)
     for _, g in frame.groupby(STATE, sort=True)]
)
strat[["unit_id", "ORISPL", "Boiler ID", "Plant Name", STATE] + POLLUTANTS].to_csv(OUT / "stratified_sample.csv", index=False)
print("Method: within each state, DataFrame.sample(n=min(5, N_h), replace=False), shared numpy Generator seeded with", SEED)
print("Total units selected:", len(strat))

Method: within each state, DataFrame.sample(n=min(5, N_h), replace=False), shared numpy Generator seeded with 2022
Total units selected: 171


**Selected unit identifiers, Part 3a.** All 171 selected units, keyed by `ORISPL` + `Boiler ID`, grouped by state. Also saved to `outputs/stratified_sample.csv`.

In [16]:
strat_ids = (strat[[STATE, "ORISPL", "Boiler ID", "unit_id", "Plant Name"]]
             .sort_values([STATE] + key).reset_index(drop=True))
strat_ids.index += 1
with pd.option_context("display.max_rows", 200):
    display(strat_ids)

,State Name,ORISPL,Boiler ID,unit_id,Plant Name
1,Alabama,3,4,3-4,Barry
2,Alabama,3,5,3-5,Barry
3,Alabama,26,5,26-5,E C Gaston
4,Alabama,6002,1,6002-1,James H Miller Jr
5,Alabama,6002,3,6002-3,James H Miller Jr
6,Arizona,160,3,160-3,Apache Station
7,Arizona,6177,U1B,6177-U1B,Coronado Generating Station
8,Arizona,8223,1,8223-1,Springerville Generating Station
9,Arizona,8223,2,8223-2,Springerville Generating Station
10,Arizona,8223,4,8223-4,Springerville Generating Station


Compact per-state view of the same selection, which is the quickest way to check a single state's draw
against its $N_h$ in the table above.

In [17]:
(strat.sort_values(key).groupby(STATE)["unit_id"]
      .agg(["count", lambda s: ", ".join(s)])
      .set_axis(["n_h", "selected ORISPL-Boiler ID"], axis=1))

,n_h,selected ORISPL-Boiler ID
State Name,,
Alabama,5,"3-4, 3-5, 26-5, 6002-1, 6002-3"
Arizona,5,"160-3, 6177-U1B, 8223-1, 8223-2, 8223-4"
Arkansas,5,"6009-1, 6138-1, 6641-1, 6641-2, 56564-SN-01"
Colorado,5,"470-1, 470-2, 470-3, 6021-C1, 6761-101"
Delaware,1,594-4
Florida,5,"136-1, 136-2, 645-BB04, 663-B2, 667-1A"
Georgia,5,"703-3BLR, 703-4BLR, 6257-1, 6257-2, 6257-3"
Illinois,5,"876-2, 879-61, 883-8, 884-4, 889-2"
Indiana,5,"983-5, 1001-2, 6113-1, 6113-4, 6137-1"


In [18]:
state_table = state_counts.join(strat[STATE].value_counts().rename("n_h (selected)"))
state_table["required"] = state_table["N_h (units in frame)"].clip(upper=5)
state_table["distinct selected"] = strat.groupby(STATE)["unit_id"].nunique()
state_table["took all units"] = state_table["N_h (units in frame)"] < 5
state_table.to_csv(OUT / "state_sample_counts.csv")
state_table

,N_h (units in frame),n_h (selected),required,distinct selected,took all units
State Name,,,,,
Alabama,7,5,5,5,False
Arizona,9,5,5,5,False
Arkansas,7,5,5,5,False
Colorado,11,5,5,5,False
Delaware,1,1,1,1,True
Florida,10,5,5,5,False
Georgia,9,5,5,5,False
Illinois,24,5,5,5,False
Indiana,32,5,5,5,False


### 3b. Check counts and Arizona results

In [19]:
assert (state_table["n_h (selected)"] == state_table["required"]).all()
assert (state_table["distinct selected"] == state_table["n_h (selected)"]).all()
assert strat["unit_id"].is_unique and strat["unit_id"].isin(frame["unit_id"]).all()
assert len(strat.drop_duplicates(subset=key)) == len(strat)   # distinct on ORISPL + Boiler ID
print("Every state has the required number of distinct eligible units:", True)
print("Distinct (ORISPL, Boiler ID) pairs selected:", len(strat.drop_duplicates(subset=key)), "of", len(strat), "rows")
print("All selected units present in the analysis frame:", strat["unit_id"].isin(frame["unit_id"]).all())
print("States with fewer than 5 units (all taken):", list(state_table.index[state_table["took all units"]]))

Every state has the required number of distinct eligible units: True
Distinct (ORISPL, Boiler ID) pairs selected: 171 of 171 rows
All selected units present in the analysis frame: True
States with fewer than 5 units (all taken): ['Delaware', 'Louisiana', 'Mississippi', 'Montana', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'South Dakota', 'Washington']


In [20]:
strat_az = strat[strat[STATE] == "Arizona"]
print("Arizona units selected:", strat_az["unit_id"].tolist())

part3b = pd.DataFrame({
    "AZ benchmark (N=%d)" % len(az): az_bench,
    "AZ, national SRS (n=%d)" % len(srs_az): srs_az[POLLUTANTS].mean() if len(srs_az) else np.nan,
    "AZ, state sample (n=%d)" % len(strat_az): strat_az[POLLUTANTS].mean(),
})
part3b["State sample - benchmark"] = part3b.iloc[:, 2] - part3b.iloc[:, 0]
part3b.index.name = "tons per unit"
part3b

Arizona units selected: ['8223-1', '160-3', '8223-4', '6177-U1B', '8223-2']


,AZ benchmark (N=9),"AZ, national SRS (n=3)","AZ, state sample (n=5)",State sample - benchmark
tons per unit,,,,
2022 SO2 (tons),808.67,969.00,968.40,159.73
2022 NOX (tons),"1,275.78","1,539.33","1,437.80",162.02
2022 CO2 (tons),"1,825,836.89","1,707,136.67","1,674,239.00","-151,597.89"


### 3c. What this design guarantees about Arizona

**What is guaranteed.** Arizona has 9 units in the frame, so the rule "take 5, or all units if fewer than 5" selects exactly 5 Arizona units with probability 1. Arizona can never be missed, and the Arizona sample size is fixed at 5 rather than being a random variable. The 2.45% chance of an empty Arizona cell from Part 2 is now zero, and every one of the 39 states is covered by construction. The small states benefit most: Delaware, South Dakota and Washington are each sampled with certainty here instead of 33% of the time. Arizona's sampling fraction also rises from roughly 1/3 to 5/9, so the estimator is more precise on average.

**What is not guaranteed.** The design does **not** guarantee that this particular sample's Arizona means are closer to the Arizona benchmarks than Part 2's were. Coverage and accuracy are different things. Which 5 of the 9 units are chosen is still random, so the Arizona mean still has sampling variation around the benchmark; a larger guaranteed n shrinks the *typical* error, but says nothing about the error of any single draw.

Our own results make this concrete. For SO2 the state sample gives 968.40 vs a benchmark of 808.67, an error of +159.73, while the Part 2 SRS gave 969.00, an error of +160.33. The two are essentially tied, and the SRS with 3 units was fractionally closer than one might expect. For CO2 the state sample is *further* from the benchmark (1,674,239, an error of -151,598) than the SRS was (1,707,137, an error of -118,700), even though it used 5 units instead of 3. For NOX the state sample is closer (1,437.80, error +162.02) than the SRS (1,539.33, error +263.56). So across the three pollutants the larger guaranteed sample won once, lost once, and tied once.

That pattern is not evidence against the design; it is what sampling variation looks like. The right way to state the claim is distributional: *across repeated draws*, the 5-unit Arizona estimator has smaller variance than a 3-unit one, and with 5 of 9 units a finite-population correction further reduces it. Any single realization can fall either way, and one draw can never adjudicate between two designs. That is the same caution Part 4c applies to the national comparison.

## Part 4: National estimates and verification
### 4a. Unweighted pooled mean of the state samples

In [21]:
pooled = strat[POLLUTANTS].mean()
part4a = pd.DataFrame({
    "US benchmark": us_bench,
    "Unweighted pooled mean (n=%d)" % len(strat): pooled,
    "Difference": pooled - us_bench,
    "% difference": 100 * (pooled - us_bench) / us_bench,
})
part4a

,US benchmark,Unweighted pooled mean (n=171),Difference,% difference
2022 SO2 (tons),"1,860.48","1,607.53",-252.95,-13.60
2022 NOX (tons),"1,293.89","1,351.17",57.28,4.43
2022 CO2 (tons),"2,111,480.54","1,991,943.80","-119,536.74",-5.66


In [22]:
# Why pooling is off: share of the frame vs share of the pooled sample, by state.
shares = pd.DataFrame({
    "frame share": state_table["N_h (units in frame)"] / len(frame),
    "pooled-sample share": state_table["n_h (selected)"] / len(strat),
})
shares["sample share / frame share"] = shares["pooled-sample share"] / shares["frame share"]
shares.sort_values("sample share / frame share")

,frame share,pooled-sample share,sample share / frame share
State Name,,,
Indiana,0.07,0.03,0.41
Kentucky,0.07,0.03,0.44
Texas,0.06,0.03,0.45
Pennsylvania,0.06,0.03,0.52
Illinois,0.05,0.03,0.55
Missouri,0.05,0.03,0.62
Wyoming,0.04,0.03,0.69
West Virginia,0.04,0.03,0.69
North Carolina,0.04,0.03,0.73


**Why covering every state does not make the pooled mean appropriate.** The pooled mean of the 171 selected units is 1,607.53 tons SO2, 1,351.17 tons NOX and 1,991,944 tons CO2, against benchmarks of 1,860.48, 1,293.89 and 2,111,481.

Covering every state solves a *coverage* problem, not a *representation* problem. The estimand is the average over the 448 units in the frame, so every unit should carry equal influence on the national estimate. Our design instead gives every **state** nearly equal influence, because almost every state contributes 5 units regardless of its size. The two are the same thing only if all states have the same number of units, and they emphatically do not here: Indiana has 32 units and Delaware has 1.

The practical effect is that the pooled sample is no longer a miniature of the frame. In the table above, Indiana is 32/448 = 7.1% of the frame but only 5/171 = 2.9% of the pooled sample, so each Indiana unit is underweighted by a factor of about 0.41. Delaware is 1/448 = 0.2% of the frame but 1/171 = 0.6% of the pooled sample, an overweight of about 2.6x. In effect, each sampled unit in a small state stands for roughly one unit, while each sampled unit in Indiana stands for 32/5 = 6.4 units, yet the unweighted mean treats them as interchangeable.

This is a **bias**, not sampling noise. Because selection probabilities differ systematically across states and the pooled mean ignores that, the estimator is centered on the wrong quantity. It targets the unweighted average of the 39 state means, which is 1,548.94 tons SO2, not the unit-level mean of 1,860.48. Our repeated-draw check in 4d confirms it: over 5,000 redraws the pooled estimator averages 1,635.90 tons SO2, persistently below the benchmark, while the weighted estimator averages 1,862.21. More data drawn the same way would not fix it; only reweighting does.

Note too that the direction of the bias depends on the pollutant and will not be uniform. SO2 is understated by 13.6% and CO2 by 5.7%, but NOX is *overstated* by 4.4%. The sign depends on whether the large states happen to be above or below average on that pollutant, which is why the error cannot be corrected with a single blanket adjustment.

### 4b. Weighted national mean

**Hand example:** State A has $N_A=100$ units with sample mean 20 tons, and State B has $N_B=10$ units with sample mean 80 tons. Five units were sampled in each state.

- Unweighted pooled mean $= \dfrac{5(20) + 5(80)}{10} = \dfrac{20+80}{2} = 50$ tons
- Weighted mean $= \dfrac{100}{110}(20) + \dfrac{10}{110}(80) = \dfrac{2000 + 800}{110} = \dfrac{2800}{110} \approx 25.45$ tons

**EPA formula:** For state $h$ with $N_h$ units in the frame, $n_h$ sampled units and sample mean $\bar y_h$:

$$w_h = \frac{N_h}{N}, \qquad N=\sum_h N_h = 448, \qquad \hat{\bar Y}_{\text{weighted}} = \sum_h w_h \, \bar y_h, \qquad \sum_h w_h = 1$$

In [23]:
# The hand example, checked in code
N_A, ybar_A, N_B, ybar_B = 100, 20, 10, 80
print("Unweighted:", (ybar_A + ybar_B) / 2)
print("Weighted:  ", round((N_A * ybar_A + N_B * ybar_B) / (N_A + N_B), 2))

Unweighted: 50.0
Weighted:   25.45


In [24]:
state_means = strat.groupby(STATE)[POLLUTANTS].mean()

weights_table = state_table[["N_h (units in frame)", "n_h (selected)"]].join(state_means)
weights_table["w_h = N_h / N"] = weights_table["N_h (units in frame)"] / weights_table["N_h (units in frame)"].sum()

weighted = pd.Series({p: (weights_table["w_h = N_h / N"] * weights_table[p]).sum() for p in POLLUTANTS})

print("Sum of normalized weights:", weights_table["w_h = N_h / N"].sum())
assert np.isclose(weights_table["w_h = N_h / N"].sum(), 1.0)

weights_table.to_csv(OUT / "state_weights_table.csv")
pd.set_option("display.float_format", "{:,.4f}".format)
display(weights_table)
pd.set_option("display.float_format", "{:,.2f}".format)
weighted.rename("Weighted national mean (tons per unit)").to_frame()

Sum of normalized weights: 1.0


,N_h (units in frame),n_h (selected),2022 SO2 (tons),2022 NOX (tons),2022 CO2 (tons),w_h = N_h / N
State Name,,,,,,
Alabama,7,5,495.2000,"1,410.4000","3,724,336.0000",0.0156
Arizona,9,5,968.4000,"1,437.8000","1,674,239.0000",0.0201
Arkansas,7,5,"4,284.0000","2,089.6000","3,119,914.6000",0.0156
Colorado,11,5,844.6000,"1,596.6000","2,332,939.0000",0.0246
Delaware,1,1,388.0000,81.0000,"169,755.0000",0.0022
Florida,10,5,"1,236.8000",595.8000,"1,685,338.4000",0.0223
Georgia,9,5,676.8000,"1,726.4000","2,587,867.8000",0.0201
Illinois,24,5,748.6000,606.4000,"1,691,208.8000",0.0536
Indiana,32,5,"1,473.0000","1,270.8000","2,004,540.0000",0.0714


,Weighted national mean (tons per unit)
2022 SO2 (tons),"1,844.96"
2022 NOX (tons),"1,322.75"
2022 CO2 (tons),"1,933,991.91"


**Treatment of states with fewer than five units.** Ten states have $N_h < 5$: Delaware (1), South Dakota (1), Washington (1), New Hampshire (2), Montana (3), Nevada (3), New Jersey (3), Louisiana (4), Mississippi (4) and New Mexico (4). For these we took every unit, so $n_h = N_h$ and the state is a **census**, not a sample. Two things follow. First, $\bar y_h$ is the state's exact mean, with zero sampling error, so these states contribute no variance to the national estimate. Second, and importantly, **they need no special weighting rule**. The weight is $w_h = N_h/N$ either way; what changes is only the per-unit design weight $d_i = N_h/n_h$, which equals 1 for a censused state and 6.4 for Indiana. Taking all units of a small state does not entitle it to extra influence, and $w_h$ correctly keeps Delaware at 1/448 = 0.0022 rather than the 1/171 = 0.0058 the pooled mean gave it.

The weights sum to 1.0 exactly, as verified above, because $\sum_h N_h = N = 448$ by construction of the frame. This is worth checking rather than assuming: it is the arithmetic guarantee that the $N_h$ we built the weights from are the post-exclusion counts, and it would fail if we had mistakenly used the raw 452-row counts for Michigan and Pennsylvania.

### 4c. Comparison of all four estimates

In [25]:
comparison = pd.DataFrame({
    "Full-frame benchmark (N=448)": us_bench,
    "National SRS (n=150)": srs_mean,
    "Stratified, unweighted (n=%d)" % len(strat): pooled,
    "Stratified, weighted (n=%d)" % len(strat): weighted,
})
comparison.index.name = "Mean tons per unit, 2022"
comparison.to_csv(OUT / "comparison_4c.csv")
comparison

,Full-frame benchmark (N=448),National SRS (n=150),"Stratified, unweighted (n=171)","Stratified, weighted (n=171)"
"Mean tons per unit, 2022",,,,
2022 SO2 (tons),"1,860.48","1,548.34","1,607.53","1,844.96"
2022 NOX (tons),"1,293.89","1,231.49","1,351.17","1,322.75"
2022 CO2 (tons),"2,111,480.54","2,006,362.91","1,991,943.80","1,933,991.91"


In [26]:
(100 * comparison.iloc[:, 1:].sub(comparison.iloc[:, 0], axis=0)
     .div(comparison.iloc[:, 0], axis=0)).rename(columns=lambda c: c + " % error")

,National SRS (n=150) % error,"Stratified, unweighted (n=171) % error","Stratified, weighted (n=171) % error"
"Mean tons per unit, 2022",,,
2022 SO2 (tons),-16.78,-13.60,-0.83
2022 NOX (tons),-4.82,4.43,2.23
2022 CO2 (tons),-4.98,-5.66,-8.41


**What the weighting corrects.** Weighting corrects a *design* distortion, nothing else. Our within-state design gave units unequal selection probabilities ($n_h/N_h$ ranges from 5/32 in Indiana to 1 in Delaware), and $w_h = N_h/N$ restores each state to its true share of the 448 units, which is what the unit-level estimand requires. The effect is large and specific: SO2 error falls from -13.6% to -0.8%, and NOX from +4.4% to +2.2%. The weighted estimate sits far closer to the benchmark on two of three pollutants.

**Why a correctly weighted estimate can still differ from the benchmark.** Weighting removes bias; it cannot remove sampling error. Even with perfect weights, we still observe only 5 of Indiana's 32 units, and $\bar y_h$ is an estimate of the state mean with its own variance. The weighted estimator is centered on the benchmark but scattered around it, and the width of that scatter is substantial: our 5,000-draw simulation in 4d gives a standard deviation of 166 tons for SO2 and 99,852 tons for CO2. Against that yardstick, the CO2 gap of -8.4% (-177,489 tons, about 1.8 SDs) and the SO2 gap of -0.8% (-15.5 tons, under 0.1 SD) are both ordinary draws, not evidence of a remaining error. Note that CO2 is the one pollutant where the weighted estimate is *further* from the benchmark than the unweighted one, which is a useful reminder that an unbiased estimator can be unlucky on a given draw.

**Do not conclude that one design is better from a single draw.** The honest comparison is between the *procedures*, judged over their sampling distributions, not between four numbers from one seed. On this draw the weighted stratified estimate beats the SRS on SO2 and NOX and loses on CO2; a different seed would reshuffle that scorecard. What we can say without appealing to luck is structural: the SRS and the weighted stratified estimator are both unbiased for the national estimand, while the unweighted pooled estimator is biased; and the stratified design additionally guarantees subgroup coverage that the SRS does not. Those claims hold across all draws. The ranking of the three numbers in this table does not.

### 4d. Verification of a consequential choice: the state weighting

The weighted estimate drives the main conclusion that the pooled stratified mean is biased and weighting fixes the bias, so we checked it three separate ways:

1. **Separate unit-level calculation.** Each sampled unit gets the design weight $d_i = N_h / n_h$, the number of frame units it stands for. We compute $\sum d_i y_i / \sum d_i$ directly from the 171 sampled rows, without the state-mean table, and check that it matches $\sum_h w_h \bar y_h$. We also check that $\sum d_i = 448$.
2. **Census check of the formula.** If we use every unit in each state (a full census), $\sum_h (N_h/N)\,\bar Y_h$ must equal the full-frame benchmark exactly, while the unweighted average of state means generally won't.
3. **Repeated sampling.** One draw can't show bias. We redraw both stratified estimators 5,000 times and compare their average with the benchmark.

In [27]:
# Check 1: unit-level design weights, computed separately from the state-mean table
chk = strat[[STATE] + POLLUTANTS].copy()
chk["d_i"] = chk[STATE].map(state_table["N_h (units in frame)"]) / chk[STATE].map(state_table["n_h (selected)"])
unit_level = pd.Series({p: (chk["d_i"] * chk[p]).sum() / chk["d_i"].sum() for p in POLLUTANTS})

print("Sum of design weights d_i:", chk["d_i"].sum(), "(should equal N = %d)" % len(frame))
pd.DataFrame({"State-level formula": weighted, "Unit-level d_i calc": unit_level,
              "Abs. difference": (weighted - unit_level).abs()})

Sum of design weights d_i: 448.0 (should equal N = 448)


,State-level formula,Unit-level d_i calc,Abs. difference
2022 SO2 (tons),"1,844.96","1,844.96",0.00
2022 NOX (tons),"1,322.75","1,322.75",0.00
2022 CO2 (tons),"1,933,991.91","1,933,991.91",0.00


In [28]:
# Check 2: census version of the formula, using every unit in each state
census_state_means = frame.groupby(STATE)[POLLUTANTS].mean()
w = state_table["N_h (units in frame)"] / len(frame)
pd.DataFrame({
    "Benchmark": us_bench,
    "Weighted avg of true state means": census_state_means.mul(w, axis=0).sum(),
    "Unweighted avg of true state means": census_state_means.mean(),
})

,Benchmark,Weighted avg of true state means,Unweighted avg of true state means
2022 SO2 (tons),"1,860.48","1,860.48","1,548.94"
2022 NOX (tons),"1,293.89","1,293.89","1,347.43"
2022 CO2 (tons),"2,111,480.54","2,111,480.54","2,113,997.25"


In [29]:
# Check 3: 5,000 repeated stratified draws (separate generator, so the submitted sample is unchanged)
sim_rng = np.random.default_rng(SEED + 1)
groups = {s: g[POLLUTANTS].to_numpy() for s, g in frame.groupby(STATE, sort=True)}
Nh = np.array([len(g) for g in groups.values()])
wh = Nh / Nh.sum()
nh = np.minimum(5, Nh)

R = 5000
unw, wtd = np.empty((R, 3)), np.empty((R, 3))
for r in range(R):
    means, rows = [], []
    for (s, g), k in zip(groups.items(), nh):
        pick = g[sim_rng.choice(len(g), size=k, replace=False)]
        rows.append(pick); means.append(pick.mean(axis=0))
    unw[r] = np.vstack(rows).mean(axis=0)
    wtd[r] = wh @ np.array(means)

pd.DataFrame({
    "Benchmark": us_bench.values,
    "Unweighted: avg over 5,000 draws": unw.mean(axis=0),
    "Weighted: avg over 5,000 draws": wtd.mean(axis=0),
    "Weighted: SD across draws": wtd.std(axis=0),
}, index=POLLUTANTS)

,Benchmark,"Unweighted: avg over 5,000 draws","Weighted: avg over 5,000 draws",Weighted: SD across draws
2022 SO2 (tons),"1,860.48","1,635.90","1,862.21",166.48
2022 NOX (tons),"1,293.89","1,344.01","1,293.72",71.05
2022 CO2 (tons),"2,111,480.54","2,137,376.26","2,113,220.89","99,851.70"


In [30]:
# Side check on another choice, the missing-data rule: what if the 4 missing CO2 values had been set to 0?
co2 = "2022 CO2 (tons)"
pd.Series({
    "CO2 mean, missing excluded (our frame, N=448)": frame[co2].mean(),
    "CO2 mean, missing set to 0 (N=452)": raw[co2].fillna(0).mean(),
    "CO2 mean, all available CO2 values (N=448)": raw[co2].mean(),
}).to_frame("tons per unit")

,tons per unit
"CO2 mean, missing excluded (our frame, N=448)","2,111,480.54"
"CO2 mean, missing set to 0 (N=452)","2,092,794.87"
"CO2 mean, all available CO2 values (N=448)","2,111,480.54"


**Did the check change our conclusion? No, it confirmed it, and it sharpened one claim.**

- *Check 1 (unit-level design weights)* reproduced the state-level weighted estimate to the last decimal on all three pollutants, with $\sum d_i = 448.0$ exactly. This rules out the most likely implementation errors: a mismatch between the $N_h$ used in the weights and the post-exclusion state counts, a groupby that silently dropped or duplicated a state, or weights built from sample counts instead of frame counts. Had we used the raw 452-row counts, $\sum d_i$ would have been 452 and the two calculations would have disagreed.
- *Check 2 (census)* confirmed the formula is algebraically right: applying $\sum_h (N_h/N)\bar Y_h$ to every unit in each state returns 1,860.48 / 1,293.89 / 2,111,480.54, identical to the benchmark, while the unweighted average of true state means returns 1,548.94 for SO2. The 20% gap between those two numbers is produced entirely by the weighting scheme and not at all by sampling, which isolates the error the weighting corrects.
- *Check 3 (5,000 redraws)* is the one that changed how we state our conclusion. Over repeated draws the weighted estimator averages 1,862.21 tons SO2 against a benchmark of 1,860.48 (and 1,293.72 vs 1,293.89 for NOX, 2,113,221 vs 2,111,481 for CO2), confirming it is unbiased, while the unweighted pooled estimator averages 1,635.90, missing low every time. So the pooled mean's error is systematic, not bad luck on our seed.

The sharpened claim concerns CO2. On our single draw the weighted estimate is 8.4% below the benchmark, *worse* than the unweighted estimate's 5.7%, which taken alone might suggest the weighting hurt. The simulation shows the weighted estimator's SD for CO2 is 99,852 tons, so our -177,489 deviation is about 1.8 SDs, well within normal variation, and the unweighted estimator's apparent advantage here is an accident of this draw. Without this check we might have reported that weighting helps SO2 and NOX but hurts CO2. The correct statement is that weighting removes bias for all three, and CO2 happened to draw unluckily. This is exactly the single-draw inference the assignment warns against, and it took a separate calculation rather than a re-reading of the same table to catch.

*We also re-examined the missing-data rule* (final cell). Excluding the 4 units gives a CO2 mean of 2,111,480.54 tons; filling the blanks with 0 gives 2,092,794.87, an 18,686-ton artifact of the fill. Since all four units reported substantial SO2 and NOX, the values are unreported rather than zero, and the exclusion rule is the defensible one. This choice also propagated into every weight, since it reduced Michigan from 19 to 17 units and Pennsylvania from 27 to 25, which is why Check 1's $\sum d_i = 448$ was worth running.

## Reproducibility and AI provenance

**Reproducibility.** Everything above runs from `data/EPA_CoalUnitCharacteristics2022.xls`, unmodified, with `SEED = 2022`. Running the notebook top to bottom regenerates every number. The selected samples are also written to `outputs/` so they stay fixed and can be checked independently of a re-run:

| File | Contents |
|---|---|
| `excluded_units.csv` | The 4 units dropped by the missing-data rule, with reason |
| `units_per_state.csv` | Frame count $N_h$ for each of the 39 states |
| `srs_sample.csv` | The 150 unit IDs in the Part 2 national SRS |
| `stratified_sample.csv` | The 171 unit IDs in the Part 3 within-state sample |
| `state_sample_counts.csv` | Part 3a table: $N_h$, $n_h$, required, distinct selected |
| `state_weights_table.csv` | Part 4b table: $N_h$, $n_h$, state sample means, $w_h$ |
| `comparison_4c.csv` | Part 4c four-estimate comparison |

Note that the 5,000-draw simulation in 4d uses a separate generator seeded `SEED + 1`, so it cannot disturb the submitted samples.

**AI provenance.** _<Team: edit this to match what you actually did, and attach the chat export.>_ AI (Claude) was used to write the Python implementation of the frame construction, sampling and weighting, and to draft the written answers. We verified its work rather than accepting it: we confirmed the 452/448 row counts and the identity of the 4 excluded units against the spreadsheet directly; we re-derived the weighted mean a second way with unit-level design weights (4d, Check 1); we confirmed the hand example in 4b by hand before running it in code; and we checked that the state counts used in the weights are the post-exclusion counts, which is what reduced Michigan to 17 and Pennsylvania to 25. The repeated-sampling check in 4d was added specifically because the single-draw CO2 result looked misleading, and it changed how we worded our conclusion.